# Advanced Features
github link: [cytozip](https://github.com/DingWB/cytozip)

This page collects advanced and lower-level workflows that go
beyond the core tutorials: 2-D region indices and on-disk
region aggregation. For DMR calling see [Calling DMRs](4.call_dmr.ipynb).


In [1]:
import os, sys
# Make sure cytozip is installed: pip install ChunkZIP
!which czip

~/Software/conda/m3c/bin/czip


## Build a region-based subset index (BED → `.cz` index)

In addition to the **context** indices (CGN / CHN, see
[2.dnam.ipynb](2.dnam.ipynb)), cytozip can build a **region** index from
any BED file. The result is a 2-D index storing, for every region in the
BED, the `(start_id, end_id)` range of reference rows that fall inside
it. This is what `czip aggregate` uses to sum mc/cov per region (e.g.
genes ± 2 kb).

Use `czip index regions` (CLI) or `cytozip.index_regions` (Python).


In [ ]:
!czip index regions --help

```shell
czip index regions -I ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz \
        -O mm10_with_chrL.allc.genes_flank2k.cz \
        -b genes_flank2k.bed.gz -j 4
```

## Aggregate genomic regions

In [3]:
!czip aggregate --help

usage: czip aggregate [-h] -I INPUT -O OUTFILE -s SSI [--intersect INTERSECT]
                      [--exclude EXCLUDE] [-c CHUNKSIZE] [-F FORMATS]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input .cz file (default: None)
  -O OUTFILE, --outfile OUTFILE
                        output .cz file (default: None)
  -s SSI, --ssi SSI     region subset index file (default: None)
  --intersect INTERSECT
                        intersect filter (default: None)
  --exclude EXCLUDE     exclude filter (default: None)
  -c CHUNKSIZE, --chunksize CHUNKSIZE
                        rows per chunk (default: 5000)
  -F FORMATS, --formats FORMATS
                        output formats (default: ['H', 'H'])


```shell
czip aggregate -I test.cz -O test_gene.cz \
        --index mm10_with_chrL.allc.genes_flank2k.cz
```

In the resulting test_gene.cz, each row is a region corresponding to one gene, mc and cov are summed up for all CG/CH located at this gene.

## DMR calling

For DMR calling cytozip exposes three callers, all consuming `.cz`
files directly:

| caller | data type | test | spatial merge |
|---|---|---|---|
| `call_dmr`        | BS-seq mc/cov, CG context | permutation RMS (ALLCools / Methylpy style) | `max_dist` + `min_dms` chaining |
| `call_dmr_ch`     | BS-seq mc/cov, CH context | permutation RMS on bin-aggregated counts + global mCH normalisation + log2fc filter | as above |
| `call_dmr_array`  | methylation array β / M (450K / EPIC / MSA) | per-probe Welch t (or Mann-Whitney) | comb-p (Stouffer-Liptak ACF + Šidák) |

For BS-seq workflows see [4.call_dmr.ipynb](4.call_dmr.ipynb).
The **array** workflow is demonstrated below.


### Calling DMRs on methylation-array `.cz`

`call_dmr_array` expects per-sample `.cz` files aligned to a probe
**reference** `.cz` (one row per probe, with a `pos` column). Each
sample `.cz` carries a single `beta` column in [0, 1] (NaN allowed).

Per-probe a two-sample test (default Welch t on M-values) gives a
p-value and Δβ; comb-p then merges adjacent significant
probes into regions and applies Šidák correction.


In [ ]:
!czip call_dmr_array --help

```shell
# CLI — EPIC two-group example
czip call_dmr_array \
    -a "ctrl/*.cz" -b "case/*.cz" \
    -r epic_manifest.cz \
    -O DMR_array.tsv \
    --test t --max_dist 1000 --jobs 16

# also dump the per-probe stats for QC / external callers
czip call_dmr_array ... --probe_pvalues_output DMR_array.probes.tsv
```

```python
# Python API
import cytozip as czip, glob

czip.call_dmr_array(
    group_a=sorted(glob.glob('ctrl/*.cz')),
    group_b=sorted(glob.glob('case/*.cz')),
    reference='epic_manifest.cz',
    output='DMR_array.tsv',
    test='t',
    sidak_p_cutoff=0.05,
    delta_beta_cutoff=0.05,
    max_dist=1000,
    jobs=16,
)
```

Output columns:

| column            | meaning |
|---|---|
| `chrom, start, end` | DMR coordinates (BED-style 0-based half-open) |
| `n_probes`         | number of probes in the region |
| `sidak_p`          | comb-p Šidák-corrected region p-value |
| `mean_delta_beta`  | mean β difference (group A − group B) over probes in the region |
| `direction`        | `'hyper'` if `mean_delta_beta > 0`, else `'hypo'` |


## Advanced Features

### Chunk Index
.cz files include a chunk index at the end of the file (magic: `CZIX`), enabling:
- **O(1) chunk lookup** by dimension name (no sequential scanning)
- **Remote reading** — only 2-3 HTTP requests needed to open and query a remote .cz file

### Remote Reading
.cz files can be read directly from HTTP/HTTPS URLs using `Reader.from_url(url)` or by passing a URL to `Reader(url)`. This uses:
- `RemoteFile` — file-like object backed by HTTP Range requests with 2MB read-ahead cache
- Chunk index — avoids scanning the entire file to find chunk boundaries
- All Reader methods (fetch, query, summary_chunks, etc.) work identically on remote files

In [ ]:
import struct, tempfile, os
import cytozip

# Example: Write .cz file with chunk index
tmpfile = tempfile.mktemp(suffix=".cz")
w = cytozip.Writer(
    output=tmpfile,
    formats=["Q", "H", "H"],
    columns=["pos", "mc", "cov"],
    dimensions=["chrom"],
)

# Write two chunks
records_chr1 = [(100, 1, 10), (200, 2, 15), (350, 3, 20)]
records_chr2 = [(50, 0, 5), (150, 1, 8), (300, 2, 12)]
w.write_chunk(b"".join(struct.pack("<QHH", *r) for r in records_chr1), ["chr1"])
w.write_chunk(b"".join(struct.pack("<QHH", *r) for r in records_chr2), ["chr2"])
w.close()

# Read and verify
r = cytozip.Reader(tmpfile)
print(f"Magic: {r.header['magic']}")

# Read chunk index
idx = r.read_chunk_index()
if idx:
    print(f"\nChunk index entries: {list(idx.keys())}")

print("\n--- chr1 records ---")
for record in r.fetch(("chr1",)):
    print(record)

print("\n--- chr2 records ---")
for record in r.fetch(("chr2",)):
    print(record)

r.close()
os.unlink(tmpfile)

2026-04-05 23:43:14.600 | DEBUG    | cytozip.cz:<module>:84 - Cython accelerated functions loaded successfully


Magic: b'CZIP'

Chunk index entries: [('chr1',), ('chr2',)]

--- chr1 records ---
[100, 1, 10]
[200, 2, 15]
[350, 3, 20]
[100, 1, 10]
[200, 2, 15]
[350, 3, 20]

--- chr2 records ---
[50, 0, 5]
[150, 1, 8]
[300, 2, 12]
[50, 0, 5]
[150, 1, 8]
[300, 2, 12]
